In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
import json

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(courses_raw)

documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:

    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

documents[0]


from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

question = "I just discovered the course. Can I join now?"
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

#search_results
[doc["question"] for doc in search_results]

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 83}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [4]:
question = "I just discovered the course. Can I join now?"
search_results = index.search(
    question,
    #boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

#search_results
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?']

In [5]:
def search(question, course="llm-zoomcamp", num_results=5):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results,
    )

search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
len(files)

72

In [7]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project